In [ ]:
from typing import List, Dict, Any


def childrenmapping(sentence):
    children = {tok["id"]: [] for tok in sentence}
    children[0] = []
    for tok in sentence:
        h = tok["head"]
        if h in children:
            children[h].append(tok["id"])
    return children


def subtreesize(node_id: int, children: Dict[int, List[int]]) -> int:
    size = 1
    for child in children.get(node_id, []):
        size += subtreesize(child, children)
    return size


def extractinterveners(
    sentences, language: str, target_deps: List[str]) -> List[Dict[str, Any]]:
    records = []

    for sentid, sentence in enumerate(sentences):
        tokbyid = {tok["id"]: tok for tok in sentence}


        children = childrenmapping(sentence)

        for tok in sentence:
            deprel = tok["deprel"]

            basedeprel = deprel.split(":")[0]

            if basedeprel not in target_deps:
                continue

            depid = tok["id"]
            headid = tok["head"]


            if headid == 0 or headid not in tokbyid:
                continue


            left = min(depid, headid)
            right = max(depid, headid)
            dep_length = right - left


            if dep_length <= 1:
                continue


            intervenerids = list(range(left + 1, right))

            for iv_id in intervenerids:
                if iv_id not in tokbyid:
                    continue
                iv_tok = tokbyid[iv_id]

                arity = len(children.get(iv_id, []))
                subsize = subtreesize(iv_id, children)
                heads_structure = subsize > 1

                records.append({
                    "language":                  language,
                    "sent_id":                   sentid,
                    "deprel":                    basedeprel,
                    "head_id":                   headid,
                    "dep_id":                    depid,
                    "dep_length":                dep_length,
                    "intervener_id":             iv_id,
                    "intervener_form":           iv_tok["form"],
                    "intervener_upos":           iv_tok["upos"],
                    "intervener_arity":          arity,
                    "intervener_subtree_size":   subsize,
                    "intervener_heads_structure": int(heads_structure),
                })

    return records
